In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw05.ipynb")

# Homework 05 — Hypothesis Testing, TVD, and Permutation Testing

**DATA2201 · Based on Lec08–09 · 100 points (80 auto-graded + 20 manually graded)**

This homework builds on concepts from **Lec08–09** using the **California–UCSD proportions**, `babyweights.csv`, and `married_couples.csv`.

Complete all **10 questions**. Questions **1–8** are graded using public and hidden Otter tests, while Questions **9–10** are graded manually based on your statistical reasoning.

Run the appropriate data-loading cells before completing the corresponding questions. Use a significance level of $\alpha = 0.05$ throughout the homework. **Do not round numerical answers unless explicitly requested.**

## Setup — Import libraries

Run the next code cell before starting the questions.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

## Part A — Hypothesis testing with a known distribution (30 points)

In Lec09, we compare the 2016 UCSD student ethnicity distribution with the California distribution. **Null hypothesis:** the UCSD distribution could result from random sampling from California. **Alternative hypothesis:** it could not be explained by random sampling alone.

Our test statistic is **Total Variation Distance (TVD)**:

$$\mathrm{TVD}(A,B)=\frac{1}{2}\sum_i |A_i-B_i|.$$

Larger TVD values indicate larger differences. These proportions are the **lecture's illustrative data**, not a current demographic estimate.

## Dataset Setup — California and UCSD Proportions

In this section, we compare the distribution of students across five ethnicity categories for **California** and **UCSD**.

The table contains:

- `California`: the proportion of students in each ethnicity category in California.
- `UCSD`: the corresponding proportion of students at UCSD.
- `N_STUDENTS = 30_000`: the sample size used later for simulation.

The five categories are **Asian, Black, Latino, White, and Other**. The proportions in each distribution sum to 1.

These values are taken from **Lec09** and will be used in the following questions on Total Variation Distance (TVD) and hypothesis testing.

Run the code cell below to create the `eth` DataFrame before continuing.

In [ ]:
eth = pd.DataFrame([
    ['Asian', 0.15, 0.51],
    ['Black', 0.05, 0.02],
    ['Latino', 0.39, 0.16],
    ['White', 0.35, 0.20],
    ['Other', 0.06, 0.11]
], columns=['Ethnicity', 'California', 'UCSD']).set_index('Ethnicity')
N_STUDENTS = 30_000
eth

### Worked Example — TVD Between Two Simple Distributions

Suppose we have two probability distributions:

$P = [0.6,\ 0.4]$

$Q = [0.5,\ 0.5]$

The Total Variation Distance (TVD) is:

$\text{TVD} = \frac{1}{2}\sum |P-Q|$

For these distributions:

$\text{TVD} = \frac{1}{2}(|0.6-0.5| + |0.4-0.5|) = 0.1$

Therefore, the TVD between the two distributions is **0.1**.

In [ ]:
example_tvd = np.abs(np.array([0.6, 0.4]) - np.array([0.5, 0.5])).sum() / 2
example_tvd

### Question 1 — Observed TVD (10 points)

Using the `eth` DataFrame:

1. Create a NumPy array named `california_probs` containing the proportions from the `California` column.
2. Create a NumPy array named `ucsd_probs` containing the proportions from the `UCSD` column.
3. Calculate the Total Variation Distance (TVD) between the two distributions and store it in `observed_eth_tvd`.

Use the TVD formula:

$\text{TVD} = \frac{1}{2}\sum |p_i-q_i|$

In [ ]:

california_probs = ...
ucsd_probs = ...
observed_eth_tvd = ...

In [ ]:
grader.check("q1")

### Worked Example — Simulating a Sample Under the Null Model

`rng.multinomial()` generates category counts from a specified probability distribution. Divide the counts by the sample size to obtain the simulated category proportions.

A fixed random seed makes the simulation reproducible.

In [ ]:
example_rng = np.random.default_rng(7)
example_counts = example_rng.multinomial(10, [0.6, 0.4])
example_counts / 10

### Question 2 — Simulate the TVD Null Distribution (10 points)

Using `np.random.default_rng(42)`:

1. Generate **300 samples**, each of size `N_STUDENTS`, from `california_probs` using `multinomial`.
2. Convert the simulated counts to proportions and store them in `eth_simulated_props`. The resulting array should have shape `(300, 5)`.
3. For each simulated sample, calculate the TVD between its category proportions and `california_probs`.
4. Store the 300 TVD values in `eth_null_tvds` as a 1D NumPy array.

In [ ]:

eth_rng = np.random.default_rng(42)
eth_simulated_props = ...
eth_null_tvds = ...

In [ ]:
grader.check("q2")

### Question 3 — TVD p-value and decision (10 points)

Using the simulated TVDs from Question 2, calculate the **simulation-based p-value** and store it in `eth_p_value`.

Then use a significance level of $\alpha = 0.05$ to determine whether the null hypothesis should be rejected. Store this decision as a Boolean in `eth_reject_null`.

**Hint:** Think about how many simulated TVDs are at least as large as the observed TVD. Then compare the resulting p-value with the significance level.

Use `eth_null_tvds` and `observed_eth_tvd` from the previous questions.

In [ ]:

eth_p_value = ...
eth_reject_null = ...

In [ ]:
grader.check("q3")

---
## Part B — Permutation Testing with Birth Weights (30 points)

In this section, you will use the birth-weight dataset to perform a **permutation test** comparing birth weights between babies of smokers and non-smokers.

The dataset contains babies' **birth weights (ounces)** and a Boolean variable indicating **maternal smoking status**.

You will use the data and concepts from Lec09 to construct the test statistic, perform the permutation procedure, and evaluate the hypothesis.

### Dataset loading — Baby weights

Run this separate code cell to load the original Lec09 CSV and retain the two columns used in the lecture.

In [ ]:
baby = pd.read_csv(Path('data') / 'babyweights.csv')
baby = baby[['Maternal Smoker', 'Birth Weight']].dropna().copy()
baby.head()

### Worked example — Difference in group means

If the smoker mean is 110 and non-smoker mean is 120, our statistic is −10.

In [ ]:
example_means = pd.Series({True: 110, False: 120})
example_means.loc[True] - example_means.loc[False]

### Question 4 — Observed Birth-Weight Difference (10 points)

Using the birth-weight data, compute the mean birth weight for each maternal smoking group and store the results in `birth_group_means`.

Next, compute the observed test statistic and store it in `observed_birth_diff`, where the test statistic is defined as:

$\text{mean birth weight of smokers} - \text{mean birth weight of non-smokers}$

**Hint:** Use an appropriate Pandas operation to summarize birth weight by maternal smoking status.

In [ ]:

birth_group_means = ...
observed_birth_diff = ...

In [ ]:
grader.check("q4")

### Worked example — Permuting the weights

Under the null hypothesis, shuffle the birth weights while keeping the smoker labels fixed. Each shuffle produces a new difference in means.

In [ ]:
example_rng = np.random.default_rng(8)
example_rng.permutation(np.array([110, 120, 130, 140]))

### Question 5 — Birth-Weight Permutation Distribution (10 points)

Construct a **permutation distribution** for the birth-weight test statistic using **300 permutations** and `np.random.default_rng(101)`.

For each permutation, randomly rearrange the birth weights while keeping the maternal smoking labels fixed. Then compute the same difference in group means defined in Question 4.

Store the 300 simulated differences in `birth_null_diffs`.

Complete the two missing expressions in the provided code.

**Hint:** A permutation should break the original relationship between birth weight and maternal smoking status while preserving the observed birth-weight values and group labels.

In [ ]:

birth_rng = np.random.default_rng(101)
birth_null_diffs = []
for _ in range(300):
    shuffled = baby.assign(Shuffled_Weight=...)
    means = shuffled.groupby('Maternal Smoker')['Shuffled_Weight'].mean()
    birth_null_diffs.append(...)

In [ ]:
grader.check("q5")

### Question 6 — Birth-Weight p-value and Decision (10 points)

Using the permutation distribution from Question 5, calculate the **one-sided p-value** for the alternative hypothesis that babies of smokers weigh less on average. Store the result in `birth_p_value`.

Then use $\alpha = 0.05$ to determine whether the null hypothesis should be rejected. Store your decision as a Boolean in `birth_reject_null`.

**Hint:** Consider which simulated differences provide evidence in the direction of the alternative hypothesis, based on how the test statistic was defined in Question 4.

In [ ]:

birth_p_value = ...
birth_reject_null = ...

In [ ]:
grader.check("q6")

---
## Part C — Permutation Testing with TVD (20 points)

In this section, you will use the employment-status data to compare the distributions of **married people** and **unmarried people living with partners**.

You will apply a **permutation test** using Total Variation Distance (TVD) to measure the difference between the two categorical distributions.

Use the concepts and procedures from Lec09 to construct the observed statistic, generate a permutation distribution, and evaluate the comparison.

### Dataset loading — Married couples

Run this separate code cell to load and label the original Lec09 dataset.

In [ ]:
couples = pd.read_csv(Path('data') / 'married_couples.csv')
empl = [
    'Working as paid employee',
    'Working, self-employed',
    'Not working - on a temporary layoff from a job',
    'Not working - looking for work',
    'Not working - retired',
    'Not working - disabled',
    'Not working - other'
]
couples = couples[['mar_status', 'empl_status', 'gender', 'age']].replace({
    'mar_status': {1: 'married', 2: 'unmarried'},
    'gender': {1: 'M', 2: 'F'},
    'empl_status': {k + 1: empl[k] for k in range(len(empl))}
}).dropna(subset=['mar_status', 'empl_status']).copy()
couples.head()

### Worked example — Comparing categorical distributions

For categorical data, compare **proportions**, not raw counts, because the two groups may have different sizes.

$$\mathrm{TVD}=\frac{1}{2}\sum_i\left|\text{proportion in group 1}-\text{proportion in group 2}\right|$$

In [ ]:
example_a = np.array([0.7, 0.3])
example_b = np.array([0.5, 0.5])
np.abs(example_a - example_b).sum() / 2

### Question 7 — Employment Distributions and Observed TVD (10 points)

Using the couples dataset, compare the **employment-status distributions** of the married and unmarried groups.

Create a table of employment-status counts for the two marital groups and store it in `employment_counts`. Convert these counts into within-group proportions and store the result in `employment_props`.

Then calculate the **Total Variation Distance (TVD)** between the two employment-status distributions and store it in `observed_couples_tvd`.

**Hint:** Think about how a pivot table can organize employment categories by marital group. Each group's proportions should form its own probability distribution.

In [ ]:

employment_counts = ...
employment_props = ...
observed_couples_tvd = ...

In [ ]:
grader.check("q7")

### Worked example — Shuffle the marital-status labels

Under the null hypothesis, shuffle marital-status labels while keeping employment categories fixed. Each shuffle gives a new TVD.

In [ ]:
example_rng = np.random.default_rng(9)
example_rng.permutation(np.array(['married', 'unmarried', 'married', 'unmarried']))

### Provided helper — TVD for two marital-status groups

Use this helper in Question 8. It calculates conditional proportions and TVD for any table containing the specified group and category columns.

In [ ]:
def tvd_of_groups(df, groups, cats):
    counts = df.pivot_table(index=cats, columns=groups, aggfunc='size', fill_value=0)
    props = counts / counts.sum()
    return (props['married'] - props['unmarried']).abs().sum() / 2

### Question 8 — Permutation TVDs and p-value (10 points)

Use **200 permutations** with `np.random.default_rng(202)` to construct a permutation distribution of TVDs for the employment-status comparison.

Use the provided `tvd_of_groups` helper function to calculate the TVD for each permutation. Store the simulated TVDs in `couples_null_tvds`.

Then calculate the simulation-based p-value in `couples_p_value` and use $\alpha = 0.05$ to store the statistical decision as a Boolean in `couples_reject_null`.

**Hint:** Under the null hypothesis, consider which group information should be shuffled while keeping the employment-status observations unchanged. For the p-value, think about whether larger or smaller TVDs provide stronger evidence that the two distributions differ.

In [ ]:

couples_rng = np.random.default_rng(202)
couples_null_tvds = []
for _ in range(200):
    shuffled = couples.assign(shuffled_mar=...)
    couples_null_tvds.append(...)
couples_p_value = ...
couples_reject_null = ...

In [ ]:
grader.check("q8")

---
## Part D — Statistical interpretation (20 points, manually graded)

Answer in your own words using the results you calculated above. Numerical results should come from your notebook, not from the worked examples.

<!-- BEGIN QUESTION -->

### Question 9 — Interpret the California–UCSD Hypothesis Test (10 points, manual)

Using your results from Questions 1–3, answer the following in **3–5 sentences total**:

1. State the **null and alternative hypotheses** for the California–UCSD comparison.
2. Report your **observed TVD** and **simulation-based p-value**, and explain what a larger TVD represents in this comparison.
3. Using $\alpha = 0.05$, state your statistical decision and explain what the result means in context.

Do **not** interpret the p-value as the probability that the null hypothesis is true.

**Manual rubric:** hypotheses (3 points), TVD/p-value and meaning (3 points), contextual conclusion (4 points).

[Write your response here.]

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Question 10 — Interpret the Two Permutation Tests (10 points, manual)

Using your results from Questions 4–8, answer the following in **4–6 sentences total**:

1. For the **birth-weight test**, report the observed difference, the one-sided permutation p-value, and your statistical decision at $\alpha = 0.05$.
2. For the **employment-status test**, report the observed TVD, the permutation p-value, and your statistical decision at $\alpha = 0.05$.
3. Compare the two permutation tests by explaining:
   - what is shuffled in each test, and
   - why the two tests use different test statistics.

**Manual rubric:** birth-weight interpretation (3 points), employment-status interpretation (3 points), permutation and test-statistic explanation (4 points).

[Write your response here.]

<!-- END QUESTION -->

---
## Submission

Run **Kernel → Restart & Run All** and verify that the public tests pass. Complete the written responses for Q9 and Q10, then submit your notebook to Gradescope as instructed. Public tests do not guarantee that all hidden tests pass.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)